In [ ]:
# - By default trains nnUNet only; set RUN_DEFORMNET=True to run steps after nnUNet

# 0) Minimal config (EDIT THESE)
KAGGLE_JSON = {"username":"cody11null","key":"913c3a99d13f0c593114f6ae9feae576"}
RUN_DEFORMNET = False  # set True to run steps after nnUNet (merge OOF, soft OOF, DeformNet, archiving)

# 1) Write kaggle.json (instead of Drive copy)
import os, sys, json, subprocess, shutil, time
from pathlib import Path

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    f.write(KAGGLE_JSON.strip())
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
!ls -l ~/.kaggle

# 2) Install deps (same simple style)
!pip install -q kaggle nnunetv2 simpleitk scikit-image tqdm pyyaml

# 3) Download data (same helpers & outputs as your snippet)
!python src/utils/kaggle_helper.py download-competition \
    --competition vesuvius-challenge-surface-detection \
    --out data/

!python src/utils/kaggle_helper.py download-dataset \
    --dataset p4rallax/vesuvius-coarse-nnunet-baseline \
    --out nnunet_results/

!pip install -r requirements.txt
!pip install -e ./src/nnunet/

# 4) Build nnUNet dataset (using src/ paths env, exactly like your code)
!NNUNet_raw=./src/nnunet/nnUNet_raw_data_base/nnUNet_raw \
 NNUNet_preprocessed=./src/nnunet/preprocessed \
 NNUNet_results=./src/nnunet/nnUNet_results \
 python src/nnUNet_utils/build_nnunet_dataset.py

# 5) Preprocess
!python src/nnUNet_utils/nnunet_preprocess.py

print('Done!')

In [ ]:
# 6) Set canonical nnUNet v2 env (lowercase)
import os
os.environ["nnUNet_raw"] = "./nnunet/nnUNet_raw_data_base/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "./nnunet/preprocessed"
os.environ["nnUNet_results"] = "./nnunet/nnUNet_results"
os.environ["NNUNET_COMPILE"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"
print(os.environ["nnUNet_raw"])
print(os.environ["nnUNet_preprocessed"])
print(os.environ["nnUNet_results"])
if "TORCH_LOGS" in os.environ:
    del os.environ["TORCH_LOGS"]

# 7) Train nnUNet (fold 0; uncomment others if needed)
!nnUNetv2_train Dataset900_VesuviusScroll 3d_fullres 0 -p nnUNetResEncUNetMPlans_30G
# !nnUNetv2_train Dataset900_VesuviusScroll 3d_fullres 1 -p nnUNetResEncUNetMPlans_30G
# !nnUNetv2_train Dataset900_VesuviusScroll 3d_fullres 2 -p nnUNetResEncUNetMPlans_30G
# !nnUNetv2_train Dataset900_VesuviusScroll 3d_fullres 3 -p nnUNetResEncUNetMPlans_30G
# !nnUNetv2_train Dataset900_VesuviusScroll 3d_fullres 4 -p nnUNetResEncUNetMPlans_30G

# 8) (OPTIONAL) Steps after nnUNet — only run if you set RUN_DEFORMNET=True above
if RUN_DEFORMNET:
    # 8a) Merge OOF softmax files
    from pathlib import Path
    import shutil
    oof = Path("./nnunet/nnUNet_results/Dataset900_VesuviusScroll/nnUNetTrainer__nnUNetResEncUNetMPlans__3d_fullres/oof_softmax")
    oof.mkdir(parents=True, exist_ok=True)
    merged = 0
    for i in range(5):
        fold = oof / f"fold{i}"
        if fold.exists():
            for f in fold.glob("*.npz"):
                f.rename(oof / f.name)
                merged += 1
            shutil.rmtree(fold, ignore_errors=True)
    print("OOF npz files merged:", merged)

    # 8b) Build nnUNet soft OOF for DeformNet3D
    env_run = os.environ.copy()
    env_run["NNUNet_raw"]          = "./nnunet/nnUNet_raw_data_base/nnUNet_raw"
    env_run["NNUNet_preprocessed"] = "./nnunet/preprocessed"
    env_run["NNUNet_results"]      = "./nnunet/nnUNet_results"
    subprocess.run([sys.executable, "nnUNet_utils/generate_nnunet_soft_oof.py"], check=True, env=env_run)

    # 8c) Write configs/config_deform.yaml
    import yaml
    Path("./configs").mkdir(parents=True, exist_ok=True)
    cfg = {
        "data_path": "./data",
        "nnunet_path": "./nnunet/nnUNet_results",
        "petrained_ckpt_path": "",
        "data_split_path": "./nnunet/nnUNet_results/Dataset900_VesuviusScroll/splits_final.json"
    }
    with open("./configs/config_deform.yaml", "w") as f:
        yaml.safe_dump(cfg, f)
    print(open("./configs/config_deform.yaml").read())

    # 8d) Train DeformNet
    subprocess.run([sys.executable, "train_deformnet.py"], check=True)

    # 8e) Archive artifacts
    import json, time
    artifacts_dir = Path("./artifacts"); artifacts_dir.mkdir(exist_ok=True)
    stamp = time.strftime("%Y%m%d-%H%M%S")
    src = Path("./nnunet/nnUNet_results")
    zip_path = artifacts_dir / f"vesuvius-models-v2_{stamp}.zip"
    shutil.make_archive(str(zip_path).replace(".zip",""), "zip", src)
    manifest = {
        "name": "vesuvius-models-v2",
        "created": stamp,
        "competition": "vesuvius-challenge-surface-detection",
        "dataset_id": 900,
        "config": "3d_fullres",
        "plans": "nnUNetResEncUNetMPlans",
        "results_dir": str(src),
        "zip": str(zip_path)
    }
    (artifacts_dir / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))
    print("Saved:", zip_path)
    print("Manifest:", (artifacts_dir / "MANIFEST.json").read_text())

print("✅ Pipeline finished (nnUNet only by default). Set RUN_DEFORMNET=True to run the optional post-nnUNet steps.")